In [47]:
import pandas as pd
import numpy as np
import tensorflow as tf 

In [48]:
from tensorflow.keras import models, layers

def build_cnn(input_shape, num_classes):
    model = models.Sequential()

    model.add(layers.Conv1D(32, 3, activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling1D(2))

    model.add(layers.Conv1D(64, 3, activation='relu'))
    model.add(layers.MaxPooling1D(2))
    
    model.add(layers.Conv1D(128, 3, activation='relu'))
    model.add(layers.MaxPooling1D(2))
    
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dense(num_classes, activation='softmax'))

    model.summary()
    return model

In [49]:
data = pd.read_csv("./dataSet.csv",index_col=0)#get data from csv
print(data.head())
label=data['label'].to_numpy()
subjects=data['subject'].to_numpy()
feature=data.drop(['subject','label'],axis=1).to_numpy()
feature=np.expand_dims(feature,axis=0)

     ACC0    Acc1    ACC2       ECG       EMG       EDA      resp       temp  \
0  0.8914 -0.1102 -0.2576  0.030945 -0.003708  5.710983  1.191711  29.083618   
1  0.8926 -0.1086 -0.2544  0.033646 -0.014145  5.719376  1.139832  29.122437   
2  0.8930 -0.1094 -0.2580  0.033005  0.010208  5.706406  1.141357  29.115234   
3  0.8934 -0.1082 -0.2538  0.031815  0.012634  5.712509  1.155090  29.126709   
4  0.8930 -0.1096 -0.2570  0.030350  0.002060  5.727005  1.133728  29.100860   

   label subject  
0      1      S2  
1      1      S2  
2      1      S2  
3      1      S2  
4      1      S2  


In [50]:
group=data.groupby(['subject','label'])#group by subject and label
print(group.size())#show groups size


subject  label
S10      1        826000
         2        507500
S11      1        826000
         2        476000
S13      1        826001
         2        464800
S14      1        826000
         2        472500
S15      1        822500
         2        480200
S16      1        826000
         2        471101
S17      1        826700
         2        506100
S2       1        800800
         2        430500
S3       1        798000
         2        448000
S4       1        810601
         2        444500
S5       1        838600
         2        451500
S6       1        826000
         2        455000
S7       1        830200
         2        448000
S8       1        818300
         2        469000
S9       1        826000
         2        451500
dtype: int64


In [51]:
keys=group.groups.keys()#get and show groups keys
windows_size=700
process_data=[]
label=[]
for i in keys:
    print(i)
    temp=group.get_group(i)
    temp_feature=temp.drop(['subject','label'],axis=1).values.tolist()
    temp_subject=temp['subject'].values.tolist()
    temp_label=temp['label'].values.tolist()
    print(len(temp))
    begin=0
    end=windows_size
    while end<len(temp):
        temp_process_data=[]
        temp_data=np.array(temp_feature[begin:end],dtype="float32")
        process_data.append(temp_data)
        label.append(temp_label[0])
        begin+=windows_size
        end+=windows_size


('S10', 1)
826000
('S10', 2)
507500
('S11', 1)
826000
('S11', 2)
476000
('S13', 1)
826001
('S13', 2)
464800
('S14', 1)
826000
('S14', 2)
472500
('S15', 1)
822500
('S15', 2)
480200
('S16', 1)
826000
('S16', 2)
471101
('S17', 1)
826700
('S17', 2)
506100
('S2', 1)
800800
('S2', 2)
430500
('S3', 1)
798000
('S3', 2)
448000
('S4', 1)
810601
('S4', 2)
444500
('S5', 1)
838600
('S5', 2)
451500
('S6', 1)
826000
('S6', 2)
455000
('S7', 1)
830200
('S7', 2)
448000
('S8', 1)
818300
('S8', 2)
469000
('S9', 1)
826000
('S9', 2)
451500


In [52]:
print(np.array(process_data).shape)
print(len(label))

(27550, 700, 8)
27550


In [53]:
models=build_cnn([700,8],2)

Model: "sequential_11"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d_3 (Conv1D)           (None, 698, 32)           800       
                                                                 
 max_pooling1d_3 (MaxPoolin  (None, 349, 32)           0         
 g1D)                                                            
                                                                 
 conv1d_4 (Conv1D)           (None, 347, 64)           6208      
                                                                 
 max_pooling1d_4 (MaxPoolin  (None, 173, 64)           0         
 g1D)                                                            
                                                                 
 conv1d_5 (Conv1D)           (None, 171, 128)          24704     
                                                                 
 max_pooling1d_5 (MaxPoolin  (None, 85, 128)         

In [ ]:
from sklearn.model_selection import train_test_split
process_data=np.array(process_data)
label=np.array(label,dtype='int8')
one_hot=tf.one_hot(label,2)
#xTrain, xTest, yTrain, yTest = train_test_split(process_data, label, test_size=0.2, random_state=302)

[1 1 1 ... 2 2 2]


In [55]:
models.compile(optimizer="adam",
              loss='binary_crossentropy',
              metrics=['accuracy'])
models.fit(xTrain,yTrain,batch_size=32,epochs=10)

Epoch 1/10


ValueError: in user code:

    File "c:\Users\ben92\AppData\Local\Programs\Python\Python38\lib\site-packages\keras\src\engine\training.py", line 1338, in train_function  *
        return step_function(self, iterator)
    File "c:\Users\ben92\AppData\Local\Programs\Python\Python38\lib\site-packages\keras\src\engine\training.py", line 1322, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "c:\Users\ben92\AppData\Local\Programs\Python\Python38\lib\site-packages\keras\src\engine\training.py", line 1303, in run_step  **
        outputs = model.train_step(data)
    File "c:\Users\ben92\AppData\Local\Programs\Python\Python38\lib\site-packages\keras\src\engine\training.py", line 1081, in train_step
        loss = self.compute_loss(x, y, y_pred, sample_weight)
    File "c:\Users\ben92\AppData\Local\Programs\Python\Python38\lib\site-packages\keras\src\engine\training.py", line 1139, in compute_loss
        return self.compiled_loss(
    File "